# BC–WA PM2.5 benchmark on Kaggle

**Settings (right panel):** Accelerator **GPU T4 x2** · Internet **On** (phone-verified account).
Then **Save Version → Save & Run All** - it runs in the background (browser can be closed) and
everything in `/kaggle/working/metrics` is saved as this version's output.

One version = one model group. Both T4s are used: the group's models (or, for a one-model group,
its 5 runs) are split over the two GPUs.

| Group | Models |
|---|---|
| 1 | AirLapseV2, **tuned for this dataset** (hyperparameter search on the validation set, then 5 runs; results labelled `AirLapseV2_tuned`) |
| 2 | STMamba, Informer, AirFormer |
| 3 | TCN_DIR, Autoformer |
| 4 | AirDDE (slow ODE model) |
| 5 | AirPhyNet (slow ODE model, 3-7.5 h per run on a T4) |
| 6 | MLP, GRU, LSTM, PM25_GNN, Crossformer, Transformer, AirDualODE |

**Group 1** first searches AirLapseV2's hyperparameters for `TUNE_HOURS` (one Optuna worker per GPU,
validation loss only - the test set is never used for tuning), then trains the best configuration
5 times. To continue an unfinished search in a later version, attach that version's output: the
search database is picked up and the search resumes. Set `TUNE_HOURS = 0` to skip searching and just
train with the best configuration found so far.

**Staying inside your GPU quota:** set `GPU_HOURS_LEFT` to what Kaggle shows as remaining. Everything
gets a hard deadline 20 min before that (and before Kaggle's 12 h limit); unfinished runs are redone
next time, finished ones are kept.

To continue a group or combine groups, add earlier versions' output as input
(**Add Input → Notebooks → this notebook → that version**); their results are picked up automatically.

In [ ]:
GROUP = 1              # 1-6, see the table above
REPEATS = 5
GPU_HOURS_LEFT = 6.0   # remaining Kaggle GPU quota this week
TUNE_HOURS = 4.0       # group 1 only: hyperparameter-search budget (0 = use the best found so far)

In [ ]:
import glob, os, shutil, subprocess, sys, time, torch
T_START = time.time()
N_GPU = torch.cuda.device_count()
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(N_GPU)] or 'NONE - set Accelerator to GPU T4 x2')

!git clone -q https://github.com/Grace-VN/BC-WA-PM2.5-Forecast.git /tmp/repo
%cd /tmp/repo
!pip install -q -r requirements.txt optuna

METRICS = '/kaggle/working/metrics'          # saved as notebook output
os.makedirs(f'{METRICS}/scores', exist_ok=True)
os.makedirs(f'{METRICS}/tuning', exist_ok=True)
os.environ['METRICS_DIR'] = METRICS
os.environ['RESULTS_DIR'] = '/tmp/results'   # large raw arrays: not saved

# resume / combine: copy score files and any tuning database from attached earlier outputs
found = glob.glob('/kaggle/input/**/scores/*.csv', recursive=True)
for f in found:
    shutil.copy(f, f'{METRICS}/scores/')
dbs = sorted(glob.glob('/kaggle/input/**/tuning/airlapsev2_bcwa.db', recursive=True), key=os.path.getsize)
if dbs:
    shutil.copy(dbs[-1], f'{METRICS}/tuning/airlapsev2_bcwa.db')
print(f'{len(found)} earlier run results picked up; tuning database: {dbs[-1] if dbs else "none (new search)"}')

def hours_left():
    # 20 min before the quota runs out or Kaggle's 12 h limit
    return min(GPU_HOURS_LEFT, 12.0) - 20 / 60 - (time.time() - T_START) / 3600

def wait(procs, logs, every=600):
    while any(p.poll() is None for p in procs):
        time.sleep(every)
        for i, fp in enumerate(logs):
            lines = [l.rstrip() for l in open(fp) if l.strip()]
            print(f'[{(time.time() - T_START) / 3600:.1f} h] {os.path.basename(fp)}:', lines[-1][:150] if lines else 'starting')

In [ ]:
# ---- group 1 only: hyperparameter search for AirLapseV2 (validation loss only)
if GROUP == 1 and TUNE_HOURS > 0:
    # leave time for the 5 final runs (~15-25 min each, split over the GPUs)
    reserve = 25 / 60 * -(-REPEATS // max(N_GPU, 1)) + 0.25
    budget = min(TUNE_HOURS, hours_left() - reserve)
    print(f'tuning for {budget:.2f} h on {max(N_GPU, 1)} worker(s)')
    procs, logs = [], []
    for i in range(max(N_GPU, 1)):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i)) if N_GPU else dict(os.environ)
        logs.append(f'{METRICS}/tuning/worker{i}.log')
        procs.append(subprocess.Popen([sys.executable, 'tune_airlapse_v2.py', '--timeout_hours', f'{budget:.3f}'],
                                      env=env, stdout=open(logs[-1], 'w'), stderr=subprocess.STDOUT))
        time.sleep(30)                       # let the first worker create the study database
    wait(procs, logs)
if GROUP == 1:
    !python tune_airlapse_v2.py --export

In [ ]:
# ---- train the group (group 1: AirLapseV2 with the tuned configuration) over both GPUs
sys.path.insert(0, '/tmp/repo')
from benchmark import GROUPS
models = GROUPS[GROUP]
extra = []
if GROUP == 1:
    extra = ['--label', 'AirLapseV2_tuned', '--overrides', f'{METRICS}/tuning/airlapsev2_best.yaml']
n = max(N_GPU, 1)
if len(models) >= n:
    streams = [dict(models=models[i::n]) for i in range(n)]
else:
    streams = [dict(models=models, repeat_ids=list(range(REPEATS))[i::n]) for i in range(n)]

deadline = hours_left()
print(f'training deadline: {deadline:.2f} h from now')
procs, logs = [], []
for i, st in enumerate(streams):
    cmd = [sys.executable, 'benchmark.py', '--models', *st['models'], '--repeats', str(REPEATS),
           '--epochs', '50', '--early_stop', '10', '--deadline_hours', f'{deadline:.3f}', *extra]
    if N_GPU:
        cmd += ['--gpu', str(i)]
    if 'repeat_ids' in st:
        cmd += ['--repeat_ids', *map(str, st['repeat_ids'])]
    print(f'stream {i}:', ' '.join(cmd[2:]))
    logs.append(f'{METRICS}/stream{i}.log')
    procs.append(subprocess.Popen(cmd, stdout=open(logs[-1], 'w'), stderr=subprocess.STDOUT))
wait(procs, logs)

In [ ]:
# everything available in this version (this group + attached outputs): mean ± std
!python benchmark.py --summary
import pandas as pd
s = pd.read_csv(f'{METRICS}/benchmark_summary.csv', index_col=0)
cols = ['RMSE', 'MAE', 'MAPE', 'RMSE_gt35.5', 'CSI@35.5', 'POD@35.5', 'FAR@35.5', 'CSI@55.5']
table = pd.DataFrame({'runs': s.n_repeats})
for c in cols:
    d = 1 if c == 'MAPE' else 3
    table[c] = [f'{m:.{d}f} ± {sd:.{d}f}' if pd.notna(sd) else f'{m:.{d}f}'
                for m, sd in zip(s[c + '_mean'], s[c + '_std'])]
table